Конкретно этот ноутбук это CoT версия обучения. <br>
TIR версия отличается только значениями параметров enable_tools, max_turns, python_timeout (True, 96, 8.0 соответственно) <br>
Shorterbetter дополнительно отличается параметром enable_shorterbetter

In [ ]:
!python -m pip install -q --upgrade pip
!python -m pip install -q tinker tinker-cookbook datasets chz jupyter_client pandas cloudpickle openai_harmony

In [ ]:
import os
import shutil
import sys
from pathlib import Path
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
os.environ["TINKER_API_KEY"] = user_secrets.get_secret("tinker")

SRC = Path("/kaggle/input/datasets/sohatiy/script-tir/tinker_dapo_tir_cot_gptoss20b.py")
DST = Path("/kaggle/working/tinker_pure_dapo_tir_gptoss20b.py")
shutil.copy(SRC, DST)
MODULE_PATH = Path("/kaggle/working/tinker_pure_dapo_tir_gptoss20b.py")
EVAL_CSV_PATH = Path("/kaggle/input/datasets/sohatiy/test-test/reference (1).csv")

SMOKE_LOG_DIR = Path("/kaggle/working/tinker_logs/smoke_dapo_tir")
MICRO_LOG_DIR = Path("/kaggle/working/tinker_logs/micro_train_dapo_tir")

SMOKE_LOG_DIR.mkdir(parents=True, exist_ok=True)
MICRO_LOG_DIR.mkdir(parents=True, exist_ok=True)

module_dir = str(MODULE_PATH.parent)
if module_dir not in sys.path:
    sys.path.insert(0, module_dir)

In [ ]:
import datasets
import pandas as pd

ds = datasets.load_dataset("ycchen/Crystal-Math-Preview", "rewritten", split="train")
df = ds.to_pandas()

print("n total:", len(df))
display(df[["medium_pass_rate", "high_pass_rate"]].describe())

for thr in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    n = int((df["medium_pass_rate"] > thr).sum())
    print(f"medium_pass_rate > {thr:.1f}: {n:5d} rows ({n / len(df):.1%})")

In [ ]:
from tinker_pure_dapo_tir_gptoss20b import (
    make_config,
    DAPOTIRConfig,
    PythonSandbox,
    load_train_rows,
    load_eval_rows,
    remove_eval_rows_from_train,
    rollout_single_trajectory,
    rollout_group,
    collect_valid_groups,
    build_training_datums,
    build_turn_datum,
    eval_avgk,
    run_training,
    CrystalMathRows,
    _make_adam_params,
    get_ppo_loss_config,
    extract_boxed_integer,
    AssistantTurnSample,
    TrajectorySample,
    ValidGroup,
    MODEL_NAME,
    RENDERER_NAME,
)

In [ ]:
import tinker

def td_to_list(x):
    if hasattr(x, "data"):
        return list(x.data)
    return list(x)

def assert_float_lists_close(actual, expected, tol=1e-6):
    assert len(actual) == len(expected), (actual, expected)
    for a, e in zip(actual, expected):
        assert abs(float(a) - float(e)) <= tol, (actual, expected)

def make_turn(tokens, logprobs=None, prompt_tokens=(101, 102, 103)):
    if logprobs is None:
        logprobs = [-0.1] * len(tokens)

    prompt = tinker.ModelInput.from_ints(list(prompt_tokens))

    return AssistantTurnSample(
        prompt=prompt,
        tokens=list(tokens),
        logprobs=list(logprobs),
        text="dummy",
        stop_reason="stop",
    )

def make_traj(reward, is_correct, assistant_tokens, turns=None, hard_truncated=False):
    if turns is None:
        turns = [make_turn(list(range(10, 10 + assistant_tokens)))]

    return TrajectorySample(
        row={"problem": "p", "answer": 1, "problem_id": "synthetic"},
        messages=[],
        assistant_turns=turns,
        predicted_answer=1 if is_correct else None,
        is_correct=bool(is_correct),
        base_reward=1.0 if is_correct else -1.0,
        length_penalty=reward - (1.0 if is_correct else -1.0),
        reward=float(reward),
        assistant_tokens=int(assistant_tokens),
        hard_truncated=hard_truncated,
        python_calls=0,
        python_errors=0,
        turns=len(turns),
        reason="final_answer",
        max_prompt_tokens=3,
    )

turn = make_turn(
    tokens=[11, 12, 13],
    logprobs=[-0.1, -0.2, -0.3],
    prompt_tokens=[101, 102],
)
d = build_turn_datum(turn=turn, scaled_advantage=0.25)

In [ ]:
import importlib
import inspect
import tinker_pure_dapo_tir_gptoss20b as m

m = importlib.reload(m)

In [ ]:
resume_cfg = m.make_config(
    log_path="/kaggle/working/tinker_logs/dapo_cot_32k",

    eval_csv_path=str(EVAL_CSV_PATH),

    seed=54,

    learning_rate=1e-6,
    warmup_rollout_steps=10,
    lora_rank=32,
    max_steps=50,

    groups_per_batch=4,
    group_size=8,
    trajectory_minibatch_size=128,
    ppo_epochs_per_rollout=1,
    eps_low=0.2,
    eps_high=0.28,

    context_window=32768,
    context_reserve=128,
    expected_max_length=26_214,
    overlong_cache=6_554,

    enable_tools=False,
    max_turns=4,
    max_concurrent_sandboxes=4,
    max_dynamic_sampling_groups=16,

    temperature=1.0,
    top_p=1.0,

    enable_ors=True,
    enable_overlong_filtering=False,

    enable_shorterbetter=False,
    shorterbetter_alpha=2.0,
    shorterbetter_beta=3.5e-5,

    enable_rollout_debug_log=True,
    debug_save_text=False,

    max_wall_time_seconds=int(10.5 * 3600),
    final_eval_wall_time_reserve_seconds=1800,

    print_every=1,
    eval_every=999999,
    milestone_eval_every=999999,
    checkpoint_every=1,

    quick_eval_samples=1,
    milestone_eval_samples=1,
    final_eval_samples=1,
    eval_limit_rows=1,
    final_eval_limit_rows=1,
)

In [ ]:
await m.run_training(resume_cfg)